# ALTO v1 — Manual validation

Notebook này tách manual review khỏi baseline chính. Nó chỉ đọc `ranking.csv` và `summary_by_proposal.csv`, tạo một subset cân bằng theo specification/proposal feasibility, sau đó so sánh hard gate và soft ranking với nhãn người chấm.


## 1. Mount Drive và cấu hình


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/alto/v1")
OUTPUT_DIR = PROJECT_DIR / "Output"
RANKING_PATH = OUTPUT_DIR / "ranking.csv"
PROPOSAL_SUMMARY_PATH = OUTPUT_DIR / "summary_by_proposal.csv"
REVIEW_PATH = OUTPUT_DIR / "manual_review_subset.csv"

MANUAL_PROPOSALS_PER_SPEC = 2
MIN_REVIEWED_ROWS = 6


## 2. Tạo hoặc đọc review subset

Mỗi specification chọn proposal ở các mức feasibility trải đều; mỗi proposal lấy candidate auto-rank tốt nhất và một candidate giữa bảng. File đã tồn tại sẽ không bị ghi đè.

`manual_valid` (0/1) đối chiếu đúng hard gate: output hợp lệ, đúng một target person, placement, scale và relation khi cần. Anatomy, pose/contact, context preservation và photorealism dùng thang 1–5; đây là các tiêu chí do reviewer chấm, không quyết định hard validity. `manual_accept` (0/1) là quyết định chất lượng cuối cùng.


In [ ]:
ranking_df = pd.read_csv(RANKING_PATH, encoding="utf-8-sig")
proposal_stats = pd.read_csv(PROPOSAL_SUMMARY_PATH, encoding="utf-8-sig")

if not REVIEW_PATH.exists():
    review_parts = []
    for spec_id, proposals in proposal_stats.groupby("spec_id"):
        proposals = proposals.sort_values("proposal_feasibility_score")
        count = min(MANUAL_PROPOSALS_PER_SPEC, len(proposals))
        indices = np.rint(np.linspace(0, len(proposals) - 1, count)).astype(int)
        for proposal_id in dict.fromkeys(proposals.iloc[indices]["proposal_id"]):
            candidates = ranking_df[
                (ranking_df["proposal_id"] == proposal_id)
                & ~ranking_df["failure_reasons"].astype("string").str.contains(
                    "output_failed|invalid_output", na=False, regex=True
                )
            ].sort_values("auto_rank")
            if candidates.empty:
                continue
            review_parts.append(candidates.iloc[[0, len(candidates) // 2]])

    if not review_parts:
        raise RuntimeError("Không có candidate hợp lệ để tạo manual review subset.")

    review_df = pd.concat(review_parts, ignore_index=True).drop_duplicates(["proposal_id", "seed"])
    review_df = review_df[[
        "proposal_id", "spec_id", "heap_rank",
        "proposal_feasibility_score", "context_type", "seed", "output_path",
        "auto_rank", "hard_valid", "soft_score", "failure_reasons",
    ]].copy()
    for column in [
        "manual_valid", "manual_anatomy", "manual_pose_contact",
        "manual_context_preservation", "manual_photorealism",
        "manual_accept", "manual_notes",
    ]:
        review_df[column] = ""
    review_df.to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")
    print("Created:", REVIEW_PATH)
else:
    review_df = pd.read_csv(REVIEW_PATH, encoding="utf-8-sig")
    required_review_columns = {
        "proposal_id", "seed", "manual_valid", "manual_accept",
        "manual_anatomy", "manual_pose_contact",
        "manual_context_preservation", "manual_photorealism",
    }
    missing_columns = sorted(required_review_columns - set(review_df.columns))
    if missing_columns:
        raise RuntimeError(
            f"manual_review_subset.csv không đúng schema hiện tại; thiếu {missing_columns}. "
            "Hãy đổi tên file cũ rồi chạy lại cell để tạo subset mới."
        )
    review_seeds = pd.to_numeric(review_df["seed"], errors="coerce")
    ranking_pairs = set(zip(
        ranking_df["proposal_id"].astype(str), pd.to_numeric(ranking_df["seed"]).astype(int)
    ))
    review_pairs = set(zip(review_df["proposal_id"].astype(str), review_seeds))
    if review_seeds.isna().any() or not review_pairs.issubset(ranking_pairs):
        raise RuntimeError(
            "manual_review_subset.csv chứa proposal/seed không có trong ranking.csv hiện tại."
        )
    print("Using existing:", REVIEW_PATH)

display(review_df.head())


## 3. Kiểm định automatic evaluator

Sau khi điền các cột `manual_*` trong CSV, chạy lại cell này. Kết quả gồm precision/recall/agreement của hard gate và Pearson/Spearman giữa soft score hoặc proposal feasibility với đánh giá thủ công.


In [ ]:
def correlation_rows(frame, predictor, outcomes, source):
    rows = []
    for outcome in outcomes:
        valid = frame[[predictor, outcome]].dropna()
        if len(valid) < 3 or valid[predictor].nunique() < 2 or valid[outcome].nunique() < 2:
            pearson = pearson_p = spearman = spearman_p = np.nan
        else:
            pearson, pearson_p = stats.pearsonr(valid[predictor], valid[outcome])
            spearman, spearman_p = stats.spearmanr(valid[predictor], valid[outcome])
        rows.append({
            "source": source, "predictor": predictor, "outcome": outcome,
            "n": len(valid), "pearson_r": pearson, "pearson_p": pearson_p,
            "spearman_rho": spearman, "spearman_p": spearman_p,
        })
    return rows


reviewed = review_df[pd.to_numeric(review_df["manual_accept"], errors="coerce").notna()].copy()
if len(reviewed) < MIN_REVIEWED_ROWS:
    print(f"Cần ít nhất {MIN_REVIEWED_ROWS} dòng đã chấm; hiện có {len(reviewed)}.")
else:
    binary_columns = ["manual_valid", "manual_accept"]
    quality_columns = [
        "manual_anatomy", "manual_pose_contact",
        "manual_context_preservation", "manual_photorealism",
    ]
    for column in binary_columns + quality_columns:
        reviewed[column] = pd.to_numeric(reviewed[column], errors="coerce")
    reviewed["manual_mean_quality"] = reviewed[quality_columns].mean(axis=1) / 5.0

    ranking_validation = pd.DataFrame(correlation_rows(
        reviewed, "soft_score", ["manual_accept", "manual_mean_quality"], "manual_review"
    ))
    ranking_validation.to_csv(
        OUTPUT_DIR / "automatic_vs_manual_validation.csv",
        index=False, encoding="utf-8-sig",
    )

    gate_reviewed = reviewed.dropna(subset=["manual_valid"])
    predicted = gate_reviewed["hard_valid"].astype(int)
    actual = gate_reviewed["manual_valid"].astype(int)
    tp = int(((predicted == 1) & (actual == 1)).sum())
    fp = int(((predicted == 1) & (actual == 0)).sum())
    fn = int(((predicted == 0) & (actual == 1)).sum())
    gate_validation = pd.DataFrame([{
        "n": len(gate_reviewed),
        "precision": tp / (tp + fp) if tp + fp else np.nan,
        "recall": tp / (tp + fn) if tp + fn else np.nan,
        "agreement": float((predicted == actual).mean()) if len(actual) else np.nan,
    }])
    gate_validation.to_csv(
        OUTPUT_DIR / "hard_gate_manual_validation.csv",
        index=False, encoding="utf-8-sig",
    )

    manual_by_proposal = reviewed.groupby("proposal_id").agg(
        manual_success_rate=("manual_accept", "mean")
    ).reset_index()
    manual_selection = proposal_stats.merge(manual_by_proposal, on="proposal_id", how="inner")
    selection_rows = correlation_rows(
        manual_selection, "proposal_feasibility_score", ["manual_success_rate"], "manual_review"
    )
    manual_selection["proposal_feasibility_percentile"] = manual_selection.groupby("spec_id")[
        "proposal_feasibility_score"
    ].rank(method="average", pct=True)
    selection_rows.extend(correlation_rows(
        manual_selection,
        "proposal_feasibility_percentile",
        ["manual_success_rate"],
        "manual_review_within_spec_normalized",
    ))
    selection_validation = pd.DataFrame(selection_rows)
    selection_validation.to_csv(
        OUTPUT_DIR / "manual_selection_correlations.csv",
        index=False, encoding="utf-8-sig",
    )

    display(gate_validation)
    display(ranking_validation)
    display(selection_validation)
